# CIC IoT 2022: Adaptive Fisher-Weighted Z Simulation with Rolling Statistics

This notebook reproduces the CIC IoT 2022 experiment used to test whether client-specific online adaptation can reduce false positives caused by benign device heterogeneity and temporal behavioral change.

### Experiment flow
1. **Load the categorized benign CSV files** and retain the network-behavior features used by the framework.
2. **Hold out approximately 30% of physical devices per device type** using a fixed random seed. Known devices build the general profiles; held-out devices act as previously unseen clients.
3. **Create chronological, disjoint partitions**: known devices use 80% for general-profile training and 20% for baseline testing, while held-out clients use 20% enrollment, 20% PSO/client tuning, and 60% untouched final streaming evaluation.
4. **Build Fisher-weighted general profiles.** Because this dataset contains benign feature CSVs only, Fisher weights measure benign domain shift between known-device training data and trusted held-out enrollment data rather than benign-versus-attack separation.
5. **Tune seven online adaptation parameters with PSO** independently for each device type using only enrollment/tuning data and controlled synthetic drift.
6. **Enroll and calibrate each held-out client**, then process the untouched final stream with the adaptive profile.
7. **Compare baseline, drift, and adaptive conditions**, apply the causal 2-of-3 evidence rule, and save FPR/runtime/storage summaries.

### Reproducibility notes
- The notebook expects the categorized CIC IoT 2022 CSV directory and `profiles.py` at the configured relative paths.
- All data partitions are chronological within each physical device.
- Device holdout and PSO use fixed random seeds.
- Fisher feature weights remain fixed during online adaptation.
- Online updates modify compact client statistics and thresholds only; complete historical feature windows are not retained.
- CIC IoT 2022 is treated as a **benign-only evaluation** here, so this notebook reports false-positive behavior rather than attack-detection performance.


## 1. Load categorized device CSVs and create disjoint splits

This section loads all usable CIC IoT 2022 feature CSVs from the categorized device directory and attaches the device type/name from the folder structure.

To evaluate generalization across physical clients, roughly **30% of devices within each device type are held out** using a fixed seed. The remaining devices are the **known/profile devices** used to build the general device-type model.

The rows are then divided chronologically inside each device:

- **Known devices:** 80% general-profile training, 20% baseline test.
- **Held-out devices:** 20% trusted enrollment, 20% PSO/client tuning, 60% untouched final stream.

Chronological splitting is important because it prevents later device behavior from leaking into earlier training or calibration stages.


In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

from profiles import DeviceTypes, ClientProfiles


# Dataset paths, deterministic seed, and the exact feature order used everywhere.
DATA_ROOT = Path("CSV files/CIC Device Type")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

FEATURE_COLUMNS = [
    "L4_tcp",
    "L4_udp",
    "L7_http",
    "L7_https",
    "protocol",
    "most_freq_prot",
    "cnt",
    "port_class_src",
    "port_class_dst",
    "most_freq_sport",
    "most_freq_dport",
    "DNS_count",
    "NTP_count",
    "ARP_count",
    "L3_ip_dst_count",
    "pck_size",
    "total_length",
    "sum_et",
    "min_et",
    "max_et",
    "med_et",
    "average_et",
    "var",
    "q1",
    "q3",
    "iqr",
    "ethernet_frame_size",
    "sum_e",
    "min_e",
    "max_e",
    "med",
    "average",
    "var_e",
    "q1_e",
    "q3_e",
    "iqr_e",
    "ttl",
    "inter_arrival_time",
]


# Load every categorized CSV and recover device type/name from its folder path.
parts = []
skipped_files = []
for file_path in sorted(DATA_ROOT.rglob("*.csv")):
    try:
        part = pd.read_csv(file_path)
        part.columns = part.columns.str.strip()
        missing = set(FEATURE_COLUMNS + ["epoch_timestamp"]) - set(part.columns)
        if missing:
            skipped_files.append((file_path.name, sorted(missing)))
            continue

        relative = file_path.relative_to(DATA_ROOT).parts
        part = part[FEATURE_COLUMNS + ["epoch_timestamp"]].copy()
        part["device_type"] = relative[0]
        part["device_name"] = relative[1]
        part["source_file"] = str(file_path.relative_to(DATA_ROOT))
        parts.append(part)
    except Exception as error:
        skipped_files.append((file_path.name, str(error)))

combined_df = pd.concat(parts, ignore_index=True)
numeric_columns = FEATURE_COLUMNS + ["epoch_timestamp"]
combined_df[numeric_columns] = combined_df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
combined_df = combined_df.dropna(
    subset=["device_type", "device_name", *numeric_columns]
).sort_values(
    ["device_name", "epoch_timestamp", "source_file"],
    kind="stable",
).reset_index(drop=True)
combined_df["position"] = combined_df.groupby("device_name").cumcount()

# Hold out about 30% of physical devices per type to simulate previously unseen clients.
rng = np.random.default_rng(RANDOM_STATE)
heldout_rows = []
for device_type, rows in combined_df.groupby("device_type", sort=False):
    devices = sorted(rows["device_name"].unique())
    if len(devices) < 2:
        continue
    number_heldout = max(1, int(np.ceil(0.30 * len(devices))))
    for device_name in rng.choice(
        devices,
        size=number_heldout,
        replace=False,
    ):
        heldout_rows.append({
            "device_type": device_type,
            "device_name": device_name,
        })

heldout_devices = pd.DataFrame(heldout_rows)
heldout_keys = set(map(tuple, heldout_devices.to_numpy()))
is_heldout = [
    (device_type, device_name) in heldout_keys
    for device_type, device_name in combined_df[
        ["device_type", "device_name"]
    ].itertuples(index=False, name=None)
]
known_df = combined_df.loc[~np.asarray(is_heldout)].copy()
heldout_df = combined_df.loc[np.asarray(is_heldout)].copy()


def chronological_parts(dataframe, fractions, names):
    """Split each device chronologically according to the requested fractions.
    
    Splitting independently within each device preserves temporal ordering and
    prevents future observations from leaking into earlier experiment stages."""
    parts = {name: [] for name in names}
    for _, rows in dataframe.groupby("device_name", sort=False):
        rows = rows.sort_values(
            ["epoch_timestamp", "source_file"],
            kind="stable",
        ).copy()
        cuts = np.floor(np.cumsum(fractions[:-1]) * len(rows)).astype(int)
        boundaries = [0, *cuts.tolist(), len(rows)]
        for name, start, stop in zip(names, boundaries[:-1], boundaries[1:]):
            parts[name].append(rows.iloc[start:stop].copy())
    return {
        name: pd.concat(items, ignore_index=True)
        for name, items in parts.items()
    }


# Create disjoint chronological partitions so future rows cannot leak backward.
known_parts = chronological_parts(
    known_df,
    fractions=[0.80, 0.20],
    names=["general_train", "baseline_test"],
)
heldout_parts = chronological_parts(
    heldout_df,
    fractions=[0.20, 0.20, 0.60],
    names=["enrollment", "tuning", "stream"],
)
general_train_df = known_parts["general_train"]
baseline_test_df = known_parts["baseline_test"]
enrollment_df = heldout_parts["enrollment"]
tuning_df = heldout_parts["tuning"]
stream_df = heldout_parts["stream"]

split_summary = pd.DataFrame([
    {"split": "general profile training", "rows": len(general_train_df)},
    {"split": "known-device baseline test", "rows": len(baseline_test_df)},
    {"split": "held-out enrollment", "rows": len(enrollment_df)},
    {"split": "held-out PSO tuning", "rows": len(tuning_df)},
    {"split": "held-out final stream", "rows": len(stream_df)},
])
display(heldout_devices)
display(split_summary)
print(f"Loaded CSV files: {len(parts)}")
print(f"Skipped CSV files: {len(skipped_files)}")


,device_type,device_name
0,Home Automation,philipshue
1,Home Automation,amazonplug
2,Home Automation,yutron1
3,Home Automation,smartboard
4,Camera,heimvisioncam
5,Camera,arlobasecam
6,Camera,amcrest
7,Camera,simcam
8,Audio,echostudio
9,Audio,nestmini


,split,rows
0,general profile training,107843
1,known-device baseline test,26969
2,held-out enrollment,18984
3,held-out PSO tuning,18988
4,held-out final stream,56973


Loaded CSV files: 651
Skipped CSV files: 0


## 2. Build Fisher-weighted general device-type profiles

For each device type, this section learns the benign per-feature mean and standard deviation, a fixed feature-weight vector, and a 99th-percentile anomaly threshold.

> **Dataset limitation:** CIC IoT 2022 provides benign feature CSVs only in this experiment. Therefore, attack-discriminative Fisher weights cannot be learned. Instead, the Fisher criterion compares known-device general-training traffic with trusted enrollment traffic from held-out devices. These weights emphasize features that distinguish **benign device/domain variation**, not features that separate attacks from benign traffic.

The weights are normalized and remain fixed throughout the online experiment. Only the client-specific means, standard deviations, and threshold are adapted later.


In [2]:
# CIC IoT 2022 contains benign CSV data only, so attack-discriminative Fisher
# weights cannot be learned for this dataset. To preserve the same lightweight
# weighted-Z structure without inventing attack labels, the Fisher criterion is
# calculated between known-device general training data and the trusted held-out
# enrollment data. These are benign domain-shift weights, not attack weights.
SCORING_PERCENTILE = 99.0
SCORE_STD_FLOOR = 0.1


def _safe_feature_stds(values, std_floor):
    """Return finite per-feature standard deviations with a minimum floor."""
    values = np.asarray(values, dtype=float)
    ddof = 1 if len(values) > 1 else 0
    stds = values.std(axis=0, ddof=ddof)
    stds = np.nan_to_num(
        stds,
        nan=std_floor,
        posinf=std_floor,
        neginf=std_floor,
    )
    return np.maximum(stds, std_floor)


def fisher_weights(benign_values, comparison_values):
    """Learn normalized Fisher separation weights without iterative training."""
    benign_values = np.asarray(benign_values, dtype=float)
    comparison_values = np.asarray(comparison_values, dtype=float)

    benign_means = benign_values.mean(axis=0)
    benign_stds = _safe_feature_stds(benign_values, SCORE_STD_FLOOR)
    comparison_means = comparison_values.mean(axis=0)
    comparison_stds = _safe_feature_stds(
        comparison_values,
        SCORE_STD_FLOOR,
    )

    pooled = np.sqrt(benign_stds**2 + comparison_stds**2)
    weights = np.abs(comparison_means - benign_means) / np.maximum(
        pooled,
        SCORE_STD_FLOOR,
    )
    weights = np.maximum(weights, np.finfo(float).eps)
    return weights / weights.mean()


def fisher_weighted_scores(values, means, stds, weights):
    """Return the Fisher-weighted mean absolute Z score."""
    values = np.asarray(values, dtype=float)
    one_row = values.ndim == 1
    values = np.atleast_2d(values)
    z_scores = np.abs(values - means) / np.maximum(
        stds,
        SCORE_STD_FLOOR,
    )
    scores = np.average(z_scores, axis=1, weights=weights)
    return float(scores[0]) if one_row else scores


def profile_arrays(profile):
    """Return profile means/stds as arrays aligned to FEATURE_COLUMNS."""
    means = np.array(
        [profile.feature_means[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    stds = np.array(
        [profile.feature_stds[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    return means, np.maximum(stds, SCORE_STD_FLOOR)


def profile_weights(profile):
    """Return profile Fisher weights as an array aligned to FEATURE_COLUMNS."""
    return np.array(
        [profile.feature_weights[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )


# Train one fixed general profile and one fixed Fisher weight vector per device type.
general_profiles = {}
FISHER_THRESHOLD_BY_TYPE = {}
profile_rows = []
weight_rows = []
pooled_enrollment_values = enrollment_df[
    FEATURE_COLUMNS
].to_numpy(dtype=float)

# Compare known-device training behavior with trusted held-out enrollment behavior.
for device_type, train_rows in general_train_df.groupby(
    "device_type",
    sort=False,
):
    train_values = train_rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    enrollment_rows = enrollment_df[
        enrollment_df["device_type"] == device_type
    ]
    validation_rows = tuning_df[
        tuning_df["device_type"] == device_type
    ]

    if len(enrollment_rows):
        comparison_values = enrollment_rows[
            FEATURE_COLUMNS
        ].to_numpy(dtype=float)
        weight_source = "held-out benign enrollment (domain-shift Fisher)"
    elif len(pooled_enrollment_values):
        comparison_values = pooled_enrollment_values
        weight_source = "pooled benign enrollment fallback"
    else:
        # This should not occur for an evaluated held-out type, but uniform
        # weights are safer than fabricating attack observations.
        comparison_values = train_values.copy()
        weight_source = "uniform fallback"

    means = train_values.mean(axis=0)
    stds = _safe_feature_stds(train_values, SCORE_STD_FLOOR)

    if weight_source == "uniform fallback":
        weights = np.ones(len(FEATURE_COLUMNS), dtype=float)
    else:
        weights = fisher_weights(train_values, comparison_values)

    train_scores = fisher_weighted_scores(
        train_values,
        means,
        stds,
        weights,
    )
    threshold = float(
        np.percentile(train_scores, SCORING_PERCENTILE)
    )

    if len(validation_rows):
        validation_scores = fisher_weighted_scores(
            validation_rows[FEATURE_COLUMNS].to_numpy(dtype=float),
            means,
            stds,
            weights,
        )
        validation_fpr = float(
            np.mean(validation_scores > threshold)
        )
    else:
        validation_fpr = np.nan

    profile = DeviceTypes(
        device_type_name=device_type,
        feature_means=dict(zip(FEATURE_COLUMNS, means)),
        feature_stds=dict(zip(FEATURE_COLUMNS, stds)),
        feature_weights=dict(zip(FEATURE_COLUMNS, weights)),
        threshold=threshold,
        std_floor=SCORE_STD_FLOOR,
    )
    general_profiles[device_type] = profile
    FISHER_THRESHOLD_BY_TYPE[device_type] = threshold

    profile_rows.append({
        "device_type": device_type,
        "scoring_method": "Fisher-Weighted Z",
        "weight_source": weight_source,
        "training_rows": len(train_rows),
        "threshold": threshold,
        "heldout_tuning_fpr": validation_fpr,
    })

    for feature, weight in zip(FEATURE_COLUMNS, weights):
        weight_rows.append({
            "device_type": device_type,
            "feature": feature,
            "fisher_weight": float(weight),
            "weight_source": weight_source,
        })

general_profile_summary = pd.DataFrame(profile_rows)
fisher_feature_weights = pd.DataFrame(weight_rows)
display(general_profile_summary)
display(
    fisher_feature_weights.sort_values(
        ["device_type", "fisher_weight"],
        ascending=[True, False],
        kind="stable",
    ).groupby("device_type", sort=False).head(5)
)


,device_type,scoring_method,weight_source,training_rows,threshold,heldout_tuning_fpr
0,Camera,Fisher-Weighted Z,held-out benign enrollment (domain-shift Fisher),83429,1.318830,0.383805
1,Home Automation,Fisher-Weighted Z,held-out benign enrollment (domain-shift Fisher),14357,1.533747,0.053763
2,Audio,Fisher-Weighted Z,held-out benign enrollment (domain-shift Fisher),10057,1.132725,0.533871


,device_type,feature,fisher_weight,weight_source
81,Audio,most_freq_prot,3.324979,held-out benign enrollment (domain-shift Fisher)
85,Audio,most_freq_sport,2.286264,held-out benign enrollment (domain-shift Fisher)
76,Audio,L4_tcp,2.135782,held-out benign enrollment (domain-shift Fisher)
82,Audio,cnt,1.980440,held-out benign enrollment (domain-shift Fisher)
77,Audio,L4_udp,1.777302,held-out benign enrollment (domain-shift Fisher)
9,Camera,most_freq_sport,3.280422,held-out benign enrollment (domain-shift Fisher)
6,Camera,cnt,3.242944,held-out benign enrollment (domain-shift Fisher)
12,Camera,NTP_count,3.078865,held-out benign enrollment (domain-shift Fisher)
14,Camera,L3_ip_dst_count,2.940393,held-out benign enrollment (domain-shift Fisher)
7,Camera,port_class_src,2.593034,held-out benign enrollment (domain-shift Fisher)


## 3. Define lightweight adaptive Fisher profile updates with rolling statistics

Each held-out client starts from its corresponding general device-type profile. The online framework then adapts only from accepted benign/drift evidence.

The implementation uses compact Welford-style rolling statistics instead of storing complete accepted feature vectors. For the active enrollment/stable/drift batch it retains only the accepted-window count, per-feature running mean, and per-feature $M_2$. Drift confirmation additionally stores one scalar anomaly score per candidate window so the threshold can be recalibrated after confirmed drift.

The main decision paths are:

- **Enrollment:** trusted early client windows update the initial profile in fixed 64-window batches.
- **Stable behavior:** low-scoring, unflagged windows are accumulated and periodically blended into the client profile.
- **Drift candidate:** moderately elevated windows are accumulated until the PSO-selected confirmation length is reached; confirmed drift updates profile statistics and the client threshold.
- **Hard/blocked anomaly:** extreme windows are excluded from adaptation to reduce self-poisoning risk.

The final binary decision uses a causal **2-of-3 evidence rule**: at least two of the most recent three raw anomaly flags must be positive.


In [3]:
# Enrollment stays fixed; PSO tunes the online adaptation controls per device type.
# Fixed safety/decision controls. PSO tunes the seven parameters defined later.
ENROLLMENT_BATCH_SIZE = 64
RELATIVE_STD_FLOOR = 0.60
DRIFT_WARNING_RATIO = 0.80
HARD_SCORE_RATIO = 3.0
THRESHOLD_ALPHA = 0.20
UPDATE_CLIP_SIGMA = 3.0

# Final causal evidence smoothing: two anomalous windows within the latest three.
EVIDENCE_WINDOW = 3
EVIDENCE_MIN_HITS = 2


def new_client_profile(client_id, device_type):
    """Initialize a client as a copy of its general device-type profile."""
    general = general_profiles[device_type]
    return ClientProfiles(
        client_id=client_id,
        device_type_name=device_type,
        feature_means=general.feature_means,
        feature_stds=general.feature_stds,
        feature_weights=general.feature_weights,
        threshold=general.threshold,
        std_floor=general.std_floor,
    )



def new_rolling_stats(score_capacity):
    """Create a compact rolling-statistics accumulator for one client.

    Only the accepted-window count, per-feature running mean, and per-feature
    running M2 values are retained. Drift confirmation additionally keeps one
    scalar anomaly score per candidate window for threshold recalibration.
    Complete feature vectors are not stored.
    """
    score_capacity = int(score_capacity)
    if score_capacity <= 0:
        raise ValueError("score_capacity must be positive.")

    return {
        "mode": None,
        "count": 0,
        "mean": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "m2": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "score_capacity": score_capacity,
        "scores": None,
    }


def reset_rolling_stats(stats):
    """Clear the current enrollment/stable/drift accumulator in place."""
    stats["mode"] = None
    stats["count"] = 0
    stats["mean"].fill(0.0)
    stats["m2"].fill(0.0)
    stats["scores"] = None


def new_state(score_capacity):
    """Create the compact online state retained for one adaptive client."""
    return {
        # Enrollment, stable, and drift accumulation are mutually exclusive,
        # so they reuse one per-client rolling-statistics accumulator.
        "rolling_stats": new_rolling_stats(score_capacity),
        "batch_updates": 0,
        "confirmed_drifts": 0,
        "blocked_extremes": 0,
    }


def effective_arrays(profile, params):
    """Blend client and general statistics according to the anchor strength."""
    client_means, client_stds = profile_arrays(profile)
    general_means, general_stds = profile_arrays(
        general_profiles[profile.device_type_name]
    )
    anchor = float(params["anchor_strength"])
    means = (1.0 - anchor) * client_means + anchor * general_means
    variances = (
        (1.0 - anchor) * client_stds**2
        + anchor * general_stds**2
        + anchor
        * (1.0 - anchor)
        * (client_means - general_means) ** 2
    )
    return means, np.sqrt(np.maximum(variances, SCORE_STD_FLOOR**2))


def score_general(profile, values):
    """Score one feature vector using a fixed general device-type profile."""
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(profile.score_window(window))


def score_client(profile, values, params):
    """Score one feature vector using the anchored adaptive client profile."""
    general = general_profiles[profile.device_type_name]
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(
        profile.score_window(
            window,
            general_profile=general,
            anchor_strength=float(params["anchor_strength"]),
        )
    )


def score_client_matrix(profile, values, params):
    """Vectorized adaptive scoring used during tuning and threshold calibration."""
    means, stds = effective_arrays(profile, params)
    return fisher_weighted_scores(
        values,
        means,
        stds,
        profile_weights(profile),
    )



def add_rolling_observation(
    stats,
    mode,
    values,
    profile,
    params,
    score=None,
    clip_values=True,
):
    """Update exact batch statistics without retaining the feature vector."""
    if stats["mode"] != mode:
        reset_rolling_stats(stats)
        stats["mode"] = mode

    observation = np.asarray(values, dtype=np.float64)
    expected_shape = (len(FEATURE_COLUMNS),)
    if observation.shape != expected_shape:
        raise ValueError(
            f"Expected {len(FEATURE_COLUMNS)} features, got {observation.shape}."
        )

    # The profile remains fixed while a batch is accumulating. Therefore,
    # clipping each accepted observation here is equivalent to clipping the
    # complete batch immediately before the former full-buffer update.
    if clip_values:
        live_means, live_stds = effective_arrays(profile, params)
        observation = np.clip(
            observation,
            live_means - UPDATE_CLIP_SIGMA * live_stds,
            live_means + UPDATE_CLIP_SIGMA * live_stds,
        )

    new_count = stats["count"] + 1
    delta = observation - stats["mean"]
    stats["mean"] += delta / new_count
    delta_after = observation - stats["mean"]
    stats["m2"] += delta * delta_after
    stats["count"] = new_count

    if mode == "drift":
        if score is None:
            raise ValueError("A scalar score is required for drift accumulation.")
        if stats["scores"] is None:
            stats["scores"] = np.empty(
                stats["score_capacity"],
                dtype=np.float64,
            )
        if new_count > stats["scores"].size:
            raise ValueError("Drift-score accumulator capacity was exceeded.")
        stats["scores"][new_count - 1] = float(score)

    return new_count


def update_profile_from_statistics(profile, stats, alpha):
    """Blend the client profile with the accumulated batch statistics."""
    if alpha <= 0.0 or stats["count"] == 0:
        return False

    batch_means = stats["mean"].copy()
    batch_variances = stats["m2"] / stats["count"]

    old_means, old_stds = profile_arrays(profile)
    new_means = (1.0 - alpha) * old_means + alpha * batch_means
    new_variances = (
        (1.0 - alpha) * old_stds**2
        + alpha * batch_variances
        + alpha * (1.0 - alpha) * (old_means - batch_means) ** 2
    )

    _, general_stds = profile_arrays(
        general_profiles[profile.device_type_name]
    )
    minimum_stds = np.maximum(
        SCORE_STD_FLOOR,
        RELATIVE_STD_FLOOR * general_stds,
    )
    new_stds = np.maximum(
        np.sqrt(np.maximum(new_variances, 0.0)),
        minimum_stds,
    )

    profile.feature_means = dict(zip(FEATURE_COLUMNS, new_means))
    profile.feature_stds = dict(zip(FEATURE_COLUMNS, new_stds))
    return True


def drift_evidence(profile, score):
    """Use only the Fisher score ratio to separate drift candidates from attacks."""
    safe = np.finfo(float).eps
    score_ratio = score / max(profile.threshold, safe)
    hard = score_ratio > HARD_SCORE_RATIO
    candidate = (
        not hard
        and score_ratio >= DRIFT_WARNING_RATIO
    )
    return candidate, hard, score_ratio



def recalibrate_threshold_from_scores(profile, scores, params):
    """Update the threshold from compact drift-candidate score statistics."""
    scores = np.asarray(scores, dtype=np.float64)
    scores = scores[np.isfinite(scores)]
    if scores.size == 0:
        return

    candidate = float(np.percentile(scores, SCORING_PERCENTILE))
    general_threshold = general_profiles[profile.device_type_name].threshold
    blended = (
        (1.0 - THRESHOLD_ALPHA) * profile.threshold
        + THRESHOLD_ALPHA * candidate
    )
    profile.threshold = float(
        np.clip(
            blended,
            general_threshold,
            float(params["threshold_ceiling"]) * general_threshold,
        )
    )



def process_observation(profile, values, params, state, trusted=False):
    """Score one window, then route it to a stable, drift, or blocked path."""
    score_start = time.perf_counter()
    score = score_client(profile, values, params)
    score_time_ms = (time.perf_counter() - score_start) * 1000
    threshold_before = float(profile.threshold)
    flagged = profile.is_anomalous(score)
    profile.window_count += 1

    candidate, hard, score_ratio = drift_evidence(profile, score)
    updated = False
    confirmed = False
    action = "no_update"
    update_start = time.perf_counter()
    stats = state["rolling_stats"]

    if trusted:
        # Enrollment is known to be benign. Preserve the former enrollment
        # behavior by accumulating its un-clipped feature statistics.
        count = add_rolling_observation(
            stats,
            "enrollment",
            values,
            profile,
            params,
            clip_values=False,
        )
        action = "enrollment_accumulator"
        if count >= ENROLLMENT_BATCH_SIZE:
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["warmup_alpha"]),
            )
            reset_rolling_stats(stats)
            action = "enrollment_batch"

    elif hard:
        reset_rolling_stats(stats)
        state["blocked_extremes"] += 1
        action = "blocked_extreme"

    elif candidate:
        count = add_rolling_observation(
            stats,
            "drift",
            values,
            profile,
            params,
            score=score,
            clip_values=True,
        )
        action = "drift_accumulator"

        if count >= int(params["confirmation_windows"]):
            confirmed_scores = stats["scores"][:stats["count"]].copy()
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["drift_alpha"]),
            )
            if updated:
                # Complete drift feature vectors are not retained. The compact
                # score sequence observed during confirmation is therefore used
                # for percentile-based threshold recalibration.
                recalibrate_threshold_from_scores(
                    profile,
                    confirmed_scores,
                    params,
                )
                state["confirmed_drifts"] += 1
                confirmed = True

            reset_rolling_stats(stats)
            action = "drift_confirmed"

    elif not flagged:
        count = add_rolling_observation(
            stats,
            "stable",
            values,
            profile,
            params,
            clip_values=True,
        )
        action = "stable_accumulator"

        if count >= int(params["stable_batch_size"]):
            updated = update_profile_from_statistics(
                profile,
                stats,
                float(params["stable_alpha"]),
            )
            reset_rolling_stats(stats)
            action = "stable_batch"

    else:
        reset_rolling_stats(stats)
        action = "blocked_anomaly"

    if updated:
        state["batch_updates"] += 1

    return {
        "score": score,
        "threshold_before": threshold_before,
        "threshold_after": float(profile.threshold),
        "flagged": bool(flagged),
        "updated": bool(updated),
        "update_action": action,
        "drift_confirmed": bool(confirmed),
        "score_ratio": float(score_ratio),
        "score_time_ms": float(score_time_ms),
        "update_time_ms": float((time.perf_counter() - update_start) * 1000),
    }



def flush_enrollment(profile, params, state):
    """Apply any partial trusted-enrollment batch left after full batches."""
    stats = state["rolling_stats"]
    if stats["mode"] != "enrollment" or stats["count"] == 0:
        return False

    updated = update_profile_from_statistics(
        profile,
        stats,
        float(params["warmup_alpha"]),
    )
    reset_rolling_stats(stats)
    state["batch_updates"] += int(updated)
    return updated


def controlled_drift(enrollment_values, tuning_values, general_stds):
    """Create a moderate broad shift used only while tuning adaptation."""
    direction = np.sign(
        np.median(tuning_values, axis=0)
        - np.median(enrollment_values, axis=0)
    )
    fallback = np.where(
        np.arange(tuning_values.shape[1]) % 2 == 0,
        1.0,
        -1.0,
    )
    direction = np.where(direction == 0.0, fallback, direction)
    return tuning_values + 0.85 * general_stds * direction


def add_evidence_flags(dataframe, group_columns, order_column):
    """Apply the causal final decision: at least two raw flags in the last three windows."""
    result = dataframe.copy()
    result["evidence_flagged"] = False
    for _, group in result.groupby(group_columns, sort=False):
        ordered = group.sort_values(order_column, kind="stable")
        hits = (
            ordered["flagged"]
            .astype(int)
            .rolling(EVIDENCE_WINDOW, min_periods=1)
            .sum()
        )
        result.loc[ordered.index, "evidence_flagged"] = (
            hits >= EVIDENCE_MIN_HITS
        ).to_numpy()
    return result


## 4. Tune the seven adaptation parameters with PSO

PSO is run independently for each general device type. It tunes:

`warmup_alpha`, `stable_alpha`, `drift_alpha`, `anchor_strength`, `confirmation_windows`, `stable_batch_size`, and `threshold_ceiling`.

A candidate is evaluated only on held-out enrollment/tuning data plus a controlled moderate drift generated from the tuning data. Because this CIC IoT 2022 experiment has no attack partition, the objective emphasizes low false-positive rates under both stable held-out behavior and controlled drift.

The untouched final 60% client stream is never used by PSO.


In [4]:
# Cache held-out enrollment/tuning arrays so PSO does not repeatedly rebuild DataFrames.
ENROLLMENT_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)
    for client, rows in enrollment_df.groupby("device_name", sort=False)
}
TUNING_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)
    for client, rows in tuning_df.groupby("device_name", sort=False)
}
TYPE_BY_CLIENT = (
    heldout_df[["device_name", "device_type"]]
    .drop_duplicates()
    .set_index("device_name")["device_type"]
    .to_dict()
)
CLIENTS_BY_TYPE = {
    device_type: sorted(
        client
        for client, client_type in TYPE_BY_CLIENT.items()
        if client_type == device_type
    )
    for device_type in general_profiles
}


# Score one candidate only on tuning-stage data; the final stream is intentionally absent.
def simulate_candidate(device_type, params):
    """Evaluate one PSO candidate on benign tuning and controlled drift only.
    
    CIC IoT 2022 has no attack CSV partition in this experiment, so the tuning
    objective minimizes stable and controlled-drift false positives without using
    attack labels."""
    stable_flags = []
    drift_flags = []

    for client_id in CLIENTS_BY_TYPE[device_type]:
        profile = new_client_profile(client_id, device_type)
        state = new_state(int(params["confirmation_windows"]))
        enrollment_values = ENROLLMENT_BY_CLIENT[client_id]
        tuning_values = TUNING_BY_CLIENT[client_id]

        for values in enrollment_values:
            process_observation(profile, values, params, state, trusted=True)
        flush_enrollment(profile, params, state)

        for values in tuning_values:
            stable_flags.append(
                process_observation(profile, values, params, state)["flagged"]
            )

        _, general_stds = profile_arrays(general_profiles[device_type])
        for values in controlled_drift(
            enrollment_values,
            tuning_values,
            general_stds,
        ):
            drift_flags.append(
                process_observation(profile, values, params, state)["flagged"]
            )

    stable_fpr = float(np.mean(stable_flags))
    drift_fpr = float(np.mean(drift_flags))
    return {
        "objective": 5.0 * stable_fpr + 3.0 * drift_fpr,
        "stable_fpr": stable_fpr,
        "controlled_drift_fpr": drift_fpr,
    }


In [5]:
# Fixed swarm settings make the optimization repeatable across runs.
PSO_SEED = 42
PSO_PARTICLES = 8
PSO_ITERATIONS = 10
PSO_INERTIA = 0.70
PSO_COGNITIVE = 1.50
PSO_SOCIAL = 1.50

# All particle coordinates are normalized to [0, 1] and decoded into these seven controls.
PARAMETER_NAMES = [
    "warmup_alpha",
    "stable_alpha",
    "drift_alpha",
    "anchor_strength",
    "confirmation_windows",
    "stable_batch_size",
    "threshold_ceiling",
]


def interpolate(value, lower, upper):
    """Map a normalized particle coordinate from [0, 1] to a linear range."""
    return lower + value * (upper - lower)


def log_interpolate(value, lower, upper):
    """Map a normalized coordinate to a log-scaled positive parameter range."""
    return 10 ** interpolate(value, np.log10(lower), np.log10(upper))


def decode_particle(position):
    """Convert one normalized PSO particle into framework hyperparameters."""
    return {
        "warmup_alpha": log_interpolate(position[0], 0.050, 0.750),
        "stable_alpha": log_interpolate(position[1], 0.00001, 0.00500),
        "drift_alpha": log_interpolate(position[2], 0.010, 0.250),
        "anchor_strength": interpolate(position[3], 0.05, 0.40),
        "confirmation_windows": int(round(interpolate(position[4], 8, 64))),
        "stable_batch_size": int(round(interpolate(position[5], 16, 128))),
        "threshold_ceiling": interpolate(position[6], 1.00, 1.60),
    }


def conservative_params():
    """Return a hand-set safe candidate that PSO must outperform to be selected."""
    return {
        "warmup_alpha": 0.250,
        "stable_alpha": 0.00001,
        "drift_alpha": 0.050,
        "anchor_strength": 0.25,
        "confirmation_windows": 32,
        "stable_batch_size": 64,
        "threshold_ceiling": 1.50,
    }


# Standard PSO loop: evaluate, update personal/global bests, then move the swarm.
def run_pso(device_type, seed):
    """Tune one device type with standard particle-swarm position/velocity updates."""
    rng = np.random.default_rng(seed)
    dimensions = len(PARAMETER_NAMES)
    positions = rng.uniform(0.0, 1.0, size=(PSO_PARTICLES, dimensions))
    velocities = rng.uniform(-0.10, 0.10, size=(PSO_PARTICLES, dimensions))
    personal_best_positions = positions.copy()
    personal_best_scores = np.full(PSO_PARTICLES, np.inf)
    global_best_position = None
    global_best_score = np.inf
    global_best_metrics = None
    history = []

    baseline_params = conservative_params()
    baseline_metrics = simulate_candidate(device_type, baseline_params)
    history.append({
        "device_type": device_type,
        "iteration": 0,
        "particle": 0,
        "candidate": "conservative",
        **baseline_params,
        **baseline_metrics,
    })

    for iteration in range(1, PSO_ITERATIONS + 1):
        for particle_index in range(PSO_PARTICLES):
            params = decode_particle(positions[particle_index])
            metrics = simulate_candidate(device_type, params)
            history.append({
                "device_type": device_type,
                "iteration": iteration,
                "particle": particle_index + 1,
                "candidate": "pso",
                **params,
                **metrics,
            })

            if metrics["objective"] < personal_best_scores[particle_index]:
                personal_best_scores[particle_index] = metrics["objective"]
                personal_best_positions[particle_index] = positions[
                    particle_index
                ].copy()

            if metrics["objective"] < global_best_score:
                global_best_score = metrics["objective"]
                global_best_position = positions[particle_index].copy()
                global_best_metrics = metrics.copy()

        print(
            f"{device_type:>20} | iteration {iteration:>2}/{PSO_ITERATIONS} | "
            f"objective={global_best_score:.6f} | "
            f"stable FPR={global_best_metrics['stable_fpr']:.3%} | "
            f"drift FPR={global_best_metrics['controlled_drift_fpr']:.3%}"
        )

        r1 = rng.random(size=(PSO_PARTICLES, dimensions))
        r2 = rng.random(size=(PSO_PARTICLES, dimensions))
        velocities = (
            PSO_INERTIA * velocities
            + PSO_COGNITIVE
            * r1
            * (personal_best_positions - positions)
            + PSO_SOCIAL
            * r2
            * (global_best_position - positions)
        )
        positions = np.clip(positions + velocities, 0.0, 1.0)

    if baseline_metrics["objective"] <= global_best_score:
        return baseline_params, baseline_metrics, "conservative", pd.DataFrame(history)

    return (
        decode_particle(global_best_position),
        global_best_metrics,
        "pso",
        pd.DataFrame(history),
    )


## 5. Enroll and calibrate held-out clients

After PSO selects one parameter set per device type, this section creates a separate adaptive profile for every held-out physical client.

Each client begins as a copy of its general device-type profile, is updated from the trusted enrollment partition, and then processes the disjoint benign tuning stream. The resulting calibrated profile/state becomes the starting point for the untouched final evaluation.


In [6]:
# Tune once per device type, then reuse the selected parameters for its held-out clients.
BEST_PARAMS_BY_TYPE = {}
pso_rows = []
pso_histories = []

for type_index, device_type in enumerate(general_profiles):
    params, metrics, selected_from, history = run_pso(
        device_type,
        PSO_SEED + type_index,
    )
    BEST_PARAMS_BY_TYPE[device_type] = params
    pso_rows.append({
        "device_type": device_type,
        "selected_from": selected_from,
        **params,
        **metrics,
    })
    pso_histories.append(history)

pso_summary = pd.DataFrame(pso_rows)
pso_history = pd.concat(pso_histories, ignore_index=True)
display(pso_summary)


calibrated_profiles = {}
calibrated_states = {}
calibration_rows = []

        # Enrollment is trusted; the following tuning stream is processed normally.
for client_id, enrollment_values in ENROLLMENT_BY_CLIENT.items():
    device_type = TYPE_BY_CLIENT[client_id]
    params = BEST_PARAMS_BY_TYPE[device_type]
    profile = new_client_profile(client_id, device_type)
    state = new_state(int(params["confirmation_windows"]))

    for values in enrollment_values:
        process_observation(profile, values, params, state, trusted=True)
    flush_enrollment(profile, params, state)

    tuning_flags = 0
    for values in TUNING_BY_CLIENT[client_id]:
        tuning_flags += int(
            process_observation(profile, values, params, state)["flagged"]
        )

    calibrated_profiles[client_id] = profile
    calibrated_states[client_id] = state
    calibration_rows.append({
        "device_name": client_id,
        "device_type": device_type,
        "tuning_windows": len(TUNING_BY_CLIENT[client_id]),
        "tuning_false_flags": tuning_flags,
        "batch_updates": state["batch_updates"],
        "confirmed_drifts": state["confirmed_drifts"],
        "blocked_extremes": state["blocked_extremes"],
    })

calibration_summary = pd.DataFrame(calibration_rows)
display(calibration_summary)


              Camera | iteration  1/10 | objective=0.006215 | stable FPR=0.035% | drift FPR=0.150%
              Camera | iteration  2/10 | objective=0.004316 | stable FPR=0.035% | drift FPR=0.086%
              Camera | iteration  3/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  4/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  5/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  6/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  7/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  8/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration  9/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
              Camera | iteration 10/10 | objective=0.000345 | stable FPR=0.000% | drift FPR=0.012%
     Home 

,device_type,selected_from,warmup_alpha,stable_alpha,drift_alpha,anchor_strength,confirmation_windows,stable_batch_size,threshold_ceiling,objective,stable_fpr,controlled_drift_fpr
0,Camera,pso,0.107892,0.005000,0.210835,0.343339,8,50,1.00000,0.000345,0.000000,0.000115
1,Home Automation,pso,0.750000,0.005000,0.250000,0.400000,8,16,1.60000,0.067204,0.005376,0.013441
2,Audio,pso,0.050000,0.002255,0.198926,0.115332,8,128,1.14246,0.095968,0.010484,0.014516


,device_name,device_type,tuning_windows,tuning_false_flags,batch_updates,confirmed_drifts,blocked_extremes
0,amazonplug,Home Automation,117,0,8,0,0
1,amcrest,Camera,3672,0,125,1,0
2,arlobasecam,Camera,4104,0,147,0,0
3,echostudio,Audio,586,8,11,0,0
4,heimvisioncam,Camera,2757,0,97,1,0
5,nestmini,Audio,654,5,20,6,0
6,philipshue,Home Automation,184,0,14,0,0
7,simcam,Camera,6843,0,242,1,0
8,smartboard,Home Automation,23,2,2,1,0
9,yutron1,Home Automation,48,0,4,0,0


## 6. Run static Fisher, drift, and adaptive Fisher tests

The benign evaluation compares three conditions:

- **Baseline:** a fixed general device-type profile on later traffic from known/profile devices.
- **Drift Simulation:** the same fixed general profile on held-out client traffic, exposing the false positives caused by client heterogeneity and temporal change when no adaptation is available.
- **Adaptive Fisher-Weighted Z:** the client-specific adaptive profile on the same held-out final stream.

For the adaptive condition, accepted benign/drift evidence may update the client profile online. All conditions are converted from raw window flags to the same causal 2-of-3 evidence decision before FPR is reported.


In [7]:
def run_static_benign(dataframe, model_name):
    """Score benign windows with fixed general profiles and apply 2-of-3 evidence."""
    rows_out = []
    for _, row in dataframe.sort_values(
        ["device_name", "position"],
        kind="stable",
    ).iterrows():
        start = time.perf_counter()
        profile = general_profiles[row["device_type"]]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        score = score_general(profile, values)
        rows_out.append({
            "model": model_name,
            "device_name": row["device_name"],
            "device_type": row["device_type"],
            "position": int(row["position"]),
            "flagged": profile.is_anomalous(score),
            "total_time_ms": (time.perf_counter() - start) * 1000,
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


def run_adaptive_benign():
    """Process the untouched benign stream with online client adaptation."""
    rows_out = []
    for _, row in stream_df.sort_values(
        ["device_name", "position"],
        kind="stable",
    ).iterrows():
        start = time.perf_counter()
        client_id = row["device_name"]
        device_type = row["device_type"]
        profile = calibrated_profiles[client_id]
        state = calibrated_states[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        adaptive = process_observation(profile, values, params, state)
        rows_out.append({
            "model": "Adaptive Fisher-Weighted Z",
            "device_name": client_id,
            "device_type": device_type,
            "position": int(row["position"]),
            "flagged": adaptive["flagged"],
            "updated": adaptive["updated"],
            "update_action": adaptive["update_action"],
            "drift_confirmed": adaptive["drift_confirmed"],
            "total_time_ms": (time.perf_counter() - start) * 1000,
        })
    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


# Evaluate the three benign conditions on their untouched test/stream partitions.
baseline_results = run_static_benign(baseline_test_df, "Baseline")
drift_results = run_static_benign(stream_df, "Drift Simulation")
adaptive_results = run_adaptive_benign()


## 7. Report and save final evidence-level results

This section summarizes the final **2-of-3 evidence-level false-positive rates** overall, by device type, and by physical device. It also reports runtime and compact numeric storage overhead for the general/client profiles and rolling drift state.

The result tables are written to `results_ciciot2022_rolling_stats/` so the final metrics can be inspected without rerunning PSO.


In [8]:
# Aggregate evidence-level FPR before producing per-type and per-device breakdowns.
overall_results = pd.DataFrame([
    {
        "dataset": "CIC IoT 2022",
        "experiment": "Baseline",
        "benign_windows": len(baseline_results),
        "fpr": float(baseline_results["evidence_flagged"].mean()),
        "detection_rate": np.nan,
    },
    {
        "dataset": "CIC IoT 2022",
        "experiment": "Drift Simulation",
        "benign_windows": len(drift_results),
        "fpr": float(drift_results["evidence_flagged"].mean()),
        "detection_rate": np.nan,
    },
    {
        "dataset": "CIC IoT 2022",
        "experiment": "Adaptive Fisher-Weighted Z",
        "benign_windows": len(adaptive_results),
        "fpr": float(adaptive_results["evidence_flagged"].mean()),
        "detection_rate": np.nan,
    },
])

device_type_results = pd.concat(
    [baseline_results, drift_results, adaptive_results],
    ignore_index=True,
).groupby(["model", "device_type"], sort=False)["evidence_flagged"].agg(
    false_flags="sum",
    total_windows="count",
    fpr="mean",
).reset_index()

# Runtime uses only the measured per-window scoring/adaptation time from the evaluation stage.
runtime_summary = pd.concat(
    [baseline_results, drift_results, adaptive_results],
    ignore_index=True,
).groupby("model", sort=False)["total_time_ms"].agg(
    average_ms="mean",
    median_ms="median",
    maximum_ms="max",
).reset_index()

# Estimate raw numeric state only; Python object/dictionary overhead is intentionally excluded.
FLOAT64_BYTES = np.dtype(np.float64).itemsize
PROFILE_NUMERIC_VALUES = 3 * len(FEATURE_COLUMNS) + 1
PROFILE_NUMERIC_BYTES = PROFILE_NUMERIC_VALUES * FLOAT64_BYTES

# Raw numeric state only. Python object and dictionary overhead is excluded.
ROLLING_STATS_BYTES = (
    1 + 2 * len(FEATURE_COLUMNS)
) * FLOAT64_BYTES
MAX_CONFIRMATION_WINDOWS = max(
    int(params["confirmation_windows"])
    for params in BEST_PARAMS_BY_TYPE.values()
)
PEAK_DRIFT_STATE_BYTES = (
    ROLLING_STATS_BYTES
    + MAX_CONFIRMATION_WINDOWS * FLOAT64_BYTES
)

storage_summary = pd.DataFrame([
    {
        "storage_scope": "general profile statistical state",
        "items": len(general_profiles),
        "bytes_per_item": PROFILE_NUMERIC_BYTES,
        "total_bytes": len(general_profiles) * PROFILE_NUMERIC_BYTES,
        "details": (
            f"{len(FEATURE_COLUMNS)} means + "
            f"{len(FEATURE_COLUMNS)} standard deviations + "
            f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
        ),
    },
    {
        "storage_scope": "client profile statistical state",
        "items": len(calibrated_profiles),
        "bytes_per_item": PROFILE_NUMERIC_BYTES,
        "total_bytes": len(calibrated_profiles) * PROFILE_NUMERIC_BYTES,
        "details": (
            f"{len(FEATURE_COLUMNS)} means + "
            f"{len(FEATURE_COLUMNS)} standard deviations + "
            f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
        ),
    },
    {
        "storage_scope": "stable rolling-statistics state",
        "items": len(calibrated_profiles),
        "bytes_per_item": ROLLING_STATS_BYTES,
        "total_bytes": len(calibrated_profiles) * ROLLING_STATS_BYTES,
        "details": (
            f"count + {len(FEATURE_COLUMNS)} running means + "
            f"{len(FEATURE_COLUMNS)} running M2 values; "
            "stable batch closes at a PSO-selected size of "
            f"{min(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())}-"
            f"{max(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())} windows across device types"
        ),
    },
    {
        "storage_scope": "peak drift rolling-statistics state",
        "items": len(calibrated_profiles),
        "bytes_per_item": PEAK_DRIFT_STATE_BYTES,
        "total_bytes": len(calibrated_profiles) * PEAK_DRIFT_STATE_BYTES,
        "details": (
            f"rolling statistics + up to {MAX_CONFIRMATION_WINDOWS} "
            "scalar drift scores; no complete feature vectors retained"
        ),
    },
])

general_profile_summary.to_csv(
    RESULTS_DIR / "scoring_summary.csv",
    index=False,
)
fisher_feature_weights.to_csv(
    RESULTS_DIR / "fisher_feature_weights.csv",
    index=False,
)
overall_results.to_csv(RESULTS_DIR / "overall_results.csv", index=False)
device_type_results.to_csv(
    RESULTS_DIR / "device_type_fpr.csv",
    index=False,
)
pso_summary.to_csv(RESULTS_DIR / "pso_summary.csv", index=False)
runtime_summary.to_csv(RESULTS_DIR / "runtime_summary.csv", index=False)
storage_summary.to_csv(RESULTS_DIR / "storage_summary.csv", index=False)
heldout_devices.to_csv(RESULTS_DIR / "heldout_devices.csv", index=False)

print(
    "Adaptive Fisher-Weighted Z results use the deployed 2-of-3 evidence decision. "
    "Detection rate is intentionally unavailable because this archive "
    "contains no compatible attack-feature dataset."
)
display(overall_results)
display(device_type_results)
display(runtime_summary)
display(storage_summary)


Adaptive Fisher-Weighted Z results use the deployed 2-of-3 evidence decision. Detection rate is intentionally unavailable because this archive contains no compatible attack-feature dataset.


,dataset,experiment,benign_windows,fpr,detection_rate
0,CIC IoT 2022,Baseline,26969,0.005784,NaN
1,CIC IoT 2022,Drift Simulation,56973,0.370807,NaN
2,CIC IoT 2022,Adaptive Fisher-Weighted Z,56973,0.000158,NaN


,model,device_type,false_flags,total_windows,fpr
0,Baseline,Camera,143,20860,0.006855
1,Baseline,Home Automation,11,3593,0.003062
2,Baseline,Audio,2,2516,0.000795
3,Drift Simulation,Home Automation,14,1117,0.012534
4,Drift Simulation,Camera,19143,52134,0.367188
5,Drift Simulation,Audio,1969,3722,0.529017
6,Adaptive Fisher-Weighted Z,Home Automation,0,1117,0.000000
7,Adaptive Fisher-Weighted Z,Camera,1,52134,0.000019
8,Adaptive Fisher-Weighted Z,Audio,8,3722,0.002149


,model,average_ms,median_ms,maximum_ms
0,Baseline,0.214495,0.2074,46.9942
1,Drift Simulation,0.212663,0.2064,0.5796
2,Adaptive Fisher-Weighted Z,0.319504,0.3088,2.0327


,storage_scope,items,bytes_per_item,total_bytes,details
0,general profile statistical state,3,920,2760,38 means + 38 standard deviations + 38 Fisher ...
1,client profile statistical state,10,920,9200,38 means + 38 standard deviations + 38 Fisher ...
2,stable rolling-statistics state,10,616,6160,count + 38 running means + 38 running M2 value...
3,peak drift rolling-statistics state,10,680,6800,rolling statistics + up to 8 scalar drift scor...
